In [4]:
import pandas as pd
import numpy as np



In [5]:
df=pd.read_csv('data/full_hourly_demand_2023.csv')

In [6]:
df

,hour,PULocationID,demand
0,2023-01-01 00:00:00,1,0.0
1,2023-01-01 00:00:00,2,0.0
2,2023-01-01 00:00:00,3,0.0
3,2023-01-01 00:00:00,4,19.0
4,2023-01-01 00:00:00,5,0.0
...,...,...,...
2303875,2023-12-31 23:00:00,261,19.0
2303876,2023-12-31 23:00:00,262,54.0
2303877,2023-12-31 23:00:00,263,97.0
2303878,2023-12-31 23:00:00,264,14.0


In [7]:
df["hour"] = pd.to_datetime(df["hour"])


In [8]:
df["hour_of_day"] = df["hour"].dt.hour


In [9]:
pivot = df.pivot_table(
    index="PULocationID",
    columns="hour_of_day",
    values="demand",
    aggfunc="mean"
).fillna(0)


In [10]:
import numpy as np

pivot_log = np.log1p(pivot)


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
pivot_scaled = scaler.fit_transform(pivot_log)


In [12]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(
    n_components=5,      # recommended
    covariance_type="full",
    random_state=42
)

cluster_labels = gmm.fit_predict(pivot_scaled)


In [13]:
cluster_labels

array([0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 3, 0,
       2, 0, 0, 1, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0,
       0, 1, 0, 2, 0, 0, 0, 2, 3, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 4, 2,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       4, 4, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 4, 3, 0, 3, 1, 3, 3, 4, 2, 2, 0, 4, 0, 0, 3, 2, 0, 0, 0, 0,
       0, 4, 0, 0, 1, 1, 1, 1, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 2, 0, 2, 0, 0, 3, 1, 4, 2, 3, 1, 0, 3, 3, 3, 3, 0, 0, 0, 0, 2,
       0, 4, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 1, 4, 2])

In [14]:
df = df.merge(pd.DataFrame({"PULocationID": pivot.index, "cluster": cluster_labels}), on="PULocationID", how="left")


In [15]:
df

,hour,PULocationID,demand,hour_of_day,cluster
0,2023-01-01 00:00:00,1,0.0,0,0
1,2023-01-01 00:00:00,2,0.0,0,0
2,2023-01-01 00:00:00,3,0.0,0,0
3,2023-01-01 00:00:00,4,19.0,0,2
4,2023-01-01 00:00:00,5,0.0,0,0
...,...,...,...,...,...
2303875,2023-12-31 23:00:00,261,19.0,23,2
2303876,2023-12-31 23:00:00,262,54.0,23,3
2303877,2023-12-31 23:00:00,263,97.0,23,1
2303878,2023-12-31 23:00:00,264,14.0,23,4


In [18]:
import os
df.to_parquet(
    "data/hourly_demand_features.parquet",
    engine="pyarrow",
    index=False
)
